# Prework - Rozdělení dat na trénovací, validační a testovací sadu (Heart Dataset)

Rozdělení dat na podmnožiny je klíčovým krokem pro trénování a objektivní vyhodnocení modelů strojového učení.

### Proč potřebujeme 3 podmnožiny?
1. **Trénovací sada (Training set)**: Data, na kterých se model učí své vnitřní parametry (váhy, koeficienty, dělící pravidla stromů).
2. **Validační sada (Validation set)**: Data použitá pro optimalizaci a ladění hyperparametrů (např. volba $k$ u k-NN, hloubka stromu, regularizační síla $\alpha$). Chrání před přeučením (*overfitting*) na trénovací sadě.
3. **Testovací sada (Testing set)**: Data, která model ani vývojář během vývoje neviděli. Slouží k finálnímu, zcela nestrannému posouzení zobecňující schopnosti modelu na reálných nových datech.

### Návrh poměru rozdělení (Proportions):
- Pro středně velký dataset s **303 řádky** navrhujeme poměr **70 % trénovací / 15 % validační / 15 % testovací**:
  - **70 % (~212 vzorků)** poskytuje dostatek dat pro učení.
  - **15 % (~45 vzorků)** pro validaci.
  - **15 % (~46 vzorků)** pro finální test.
- Zároveň použijeme **stratifikaci** (`stratify`), aby ve všech třech sadách zůstal zachován poměr diagnóz (~54 % zdravých ku ~46 % se srdečním onemocněním).

## 1. Načtení datasetu po normalizaci

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Načtení normalizovaného datasetu
input_path = os.path.join("data", "heart_data_normalized.csv")
df = pd.read_csv(input_path)

print(f"Celkové rozměry datasetu: {df.shape[0]} řádků, {df.shape[1]} sloupců")
df.head()

## 2. Rozdělení datasetu na 3 podmnožiny (70 % / 15 % / 15 %)

Provedeme dvoustupňový split pomocí `train_test_split`:
1. Nejprve oddělíme **70 % pro trénovací sadu** a 30 % dočasnou množinu.
2. Dočasnou množinu následně rozdělíme v poměru 50:50 na **15 % validační** a **15 % testovací** sadu.

In [ ]:
# Krok 1: 70 % train, 30 % temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["ahd_yes"]
)

# Krok 2: rozdělení temp na 15 % valid a 15 % test
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["ahd_yes"]
)

## 3. Zobrazení informací o velikostech a rozložení tříd v jednotlivých sadách

In [ ]:
total = len(df)
subsets = [
    ("Trénovací sada (Train)", train_df),
    ("Validační sada (Valid)", valid_df),
    ("Testovací sada (Test)", test_df)
]

for name, subset in subsets:
    r, c = subset.shape
    pct = (r / total) * 100
    pos = (subset["ahd_yes"] == 1).sum()
    neg = (subset["ahd_yes"] == 0).sum()
    print(f"{name:25s}: {r:3d} řádků ({pct:5.1f} %), {c:2d} sloupců | ahd_yes: 0={neg:2d}, 1={pos:2d} ({pos/r*100:.1f} %)")

print(f"\nKontrolní součet řádků: {len(train_df)} + {len(valid_df)} + {len(test_df)} = {len(train_df) + len(valid_df) + len(test_df)}")

## 4. Uložení jednotlivých sad do tří samostatných .csv souborů

Při ukládání vynecháme indexový sloupec pomocí parametru `index=False`:
- `heart_data_train.csv`
- `heart_data_valid.csv`
- `heart_data_test.csv`

In [ ]:
train_path = os.path.join("data", "heart_data_train.csv")
valid_path = os.path.join("data", "heart_data_valid.csv")
test_path = os.path.join("data", "heart_data_test.csv")

train_df.to_csv(train_path, index=False)
valid_df.to_csv(valid_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Trénovací sada uložena do:  '{train_path}'")
print(f"Validační sada uložena do:  '{valid_path}'")
print(f"Testovací sada uložena do:  '{test_path}'")